In [1]:
from kafka import KafkaConsumer
import json
import matplotlib.pyplot as plt
import time

In [2]:
# Initialize data storage for plotting
timestamps = []
water_temperatures = []
ph_levels = []
turbidities = []
dissolved_oxygen_levels = []

In [3]:
# Kafka configuration
def initialize_consumer():
    kafka_topic = "water_quality"
    kafka_bootstrap_servers = ["localhost:9092"]

    # Create Kafka consumer
    consumer = KafkaConsumer(
        kafka_topic,
        bootstrap_servers=kafka_bootstrap_servers,
        value_deserializer=lambda m: json.loads(m.decode('utf-8')),
        auto_offset_reset='latest',
        enable_auto_commit=True
    )
    return consumer

In [4]:
# Receive all published messages and update plot
def update_plot(consumer):
    try:
        for message in consumer:
            # Parse the message
            sensor_data = message.value
            print(f"Received: {sensor_data}")

            # Update data storage
            timestamps.append(sensor_data['timestamp'])
            water_temperatures.append(sensor_data['water_temperature'])
            ph_levels.append(sensor_data['ph_level'])
            turbidities.append(sensor_data['turbidity'])
            dissolved_oxygen_levels.append(sensor_data['dissolved_oxygen'])

            # Keep only the last 100 entries for plotting
            if len(timestamps) > 100:
                timestamps.pop(0)
                water_temperatures.pop(0)
                ph_levels.pop(0)
                turbidities.pop(0)
                dissolved_oxygen_levels.pop(0)

            # Clear the current axes and redraw the plots
            plt.figure(figsize=(10, 8))

            plt.subplot(2, 2, 1)
            plt.plot(timestamps, water_temperatures, label="Water Temperature", color="blue")
            plt.title("Water Temperature")
            plt.ylabel("°C")

            plt.subplot(2, 2, 2)
            plt.plot(timestamps, ph_levels, label="pH Level", color="green")
            plt.title("pH Level")
            plt.ylabel("pH")

            plt.subplot(2, 2, 3)
            plt.plot(timestamps, turbidities, label="Turbidity", color="orange")
            plt.title("Turbidity")
            plt.ylabel("NTU")

            plt.subplot(2, 2, 4)
            plt.plot(timestamps, dissolved_oxygen_levels, label="Dissolved Oxygen", color="red")
            plt.title("Dissolved Oxygen")
            plt.ylabel("mg/L")

            plt.tight_layout()

            # Save the plot as an image
            plt.savefig(f"/opt/local/water_quality_plot.png")
            plt.close()

            break  # Process one message at a time
    except KeyboardInterrupt:
        print("Stopped consuming messages.")
        consumer.close()

In [5]:
# consumer = initialize_consumer()
# try:
#     while True:
#         update_plot(consumer)
# except KeyboardInterrupt:
#     print("Stopped visualization.")
# finally:
#     consumer.close()    

# 🤖 Task 3: ML

## Funciones para realiar la prediccion

In [6]:
import joblib
import pandas as pd
MODEL_PATH = "aeration_classifier.joblib"

def load_prediction_model(model_uri):
    """Carga el modelo desde el archivo local."""
    try:
        model = joblib.load(model_uri)
        print(f":D Modelo cargado exitosamente desde {model_uri}")
        return model
    except Exception as e:
        print(f":( Error al cargar el modelo: {e}")
        return None

def perform_inference(model, sensor_data):
    """
    Realiza la predicción y calcula las probabilidades basadas en los 4 parámetros.
    Retorna: (prediccion, probabilidades)
    """
    try:
        # Extraemos los datos del diccionario
        input_data = [
            sensor_data["water_temperature"], 
            sensor_data["ph_level"], 
            sensor_data["turbidity"], 
            sensor_data["dissolved_oxygen"]
        ]
        
        # 1. Predicción de la clase (ej: 0 o 1)
        prediction = model.predict([input_data])[0]
        
        # 2. Probabilidades de las clases (ej: [0.15, 0.85])
        probabilities = model.predict_proba([input_data])[0]
        
        return prediction, probabilities

    except Exception as e:
        print(f"[Error] durante la inferencia: {e}")
        return None, None

## Productor de la probabilidad de prediccion
Debe publicar:
- Resultados de inferencia
- probabilidad de pertenencia a la clase
- Imprimir por pantalla

In [7]:
from kafka import KafkaProducer

In [8]:
# Definimos un nuevo Topic donde se publicaran los resultados de inferencia
pred_topic = "prediction_topic"
kafka_bootstrap_servers = ["localhost:9092"] 

producer = KafkaProducer(
    bootstrap_servers=kafka_bootstrap_servers,
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)
print(f"Producing messages to Kafka topic '{pred_topic}'...")



Producing messages to Kafka topic 'prediction_topic'...


# Main
En este bloque de codigo ocurre todo el flujo de datos: 
1. El consumer recibe los datos y los llama al plot (como se hacia originalmente)
2. El consumer llama al modelo y realiza una prediccion
3. La prediccion se envia al nuevo topic

In [ ]:
import multiprocessing
from datetime import datetime
import time

def run_inference_worker(sensor_id, model_path, topic):
    # model = load_prediction_model(model_path)
    # consumer = initialize_consumer()
    # producer = initialize_producer()
    
    print(f"Started worker for sensor: {sensor_id}")
    
    try:
        while True:
            # Lógica de consumo simulada o real
            # for message in consumer:
            time.sleep(2) 
            time_stp = time.time()
            probas = [0.1, 0.9] # Ejemplo
            prediction = 1
            
            dt = datetime.fromtimestamp(time_stp)
            formatted_time = dt.strftime("%Y-%m-%d %H:%M:%S")
            label = "AERATION" if prediction == 1 else "NO AERATION"

            print(
                f"[{formatted_time}] | ID: {sensor_id} | "
                f"Pred: {label} ({prediction}) | "
                f"NA: {probas[0]:.2%} / A: {probas[1]:.2%}"
            )

            send_data = {
                "sensor": sensor_id,
                "timestamp": time_stp,
                "prediction": int(prediction),
                "proba_no_areation": float(probas[0]),
                "proba_areation": float(probas[1])
            }
            # producer.send("water_predictions", send_data)

    except KeyboardInterrupt:
        print(f"Worker {sensor_id} stopped.")

SENSOR_KEYS = ["SENSOR_01", "SENSOR_02", "SENSOR_03"]
MODEL_PATH = "model.h5"
processes = []

for key in SENSOR_KEYS:
    p = multiprocessing.Process(
        target=run_inference_worker, 
        args=(key, MODEL_PATH, "water_quality")
    )
    p.start()
    processes.append(p)

try:
    for p in processes:
        p.join()
except KeyboardInterrupt:
    for p in processes:
        p.terminate()

Started worker for sensor: SENSOR_01
Started worker for sensor: SENSOR_02
Started worker for sensor: SENSOR_03
[2026-03-20 17:14:54] | ID: SENSOR_01 | Pred: AERATION (1) | NA: 10.00% / A: 90.00%
[2026-03-20 17:14:54] | ID: SENSOR_02 | Pred: AERATION (1) | NA: 10.00% / A: 90.00%
[2026-03-20 17:14:54] | ID: SENSOR_03 | Pred: AERATION (1) | NA: 10.00% / A: 90.00%
[2026-03-20 17:14:56] | ID: SENSOR_01 | Pred: AERATION (1) | NA: 10.00% / A: 90.00%
[2026-03-20 17:14:56] | ID: SENSOR_02 | Pred: AERATION (1) | NA: 10.00% / A: 90.00%
[2026-03-20 17:14:56] | ID: SENSOR_03 | Pred: AERATION (1) | NA: 10.00% / A: 90.00%
[2026-03-20 17:14:58] | ID: SENSOR_01 | Pred: AERATION (1) | NA: 10.00% / A: 90.00%
[2026-03-20 17:14:58] | ID: SENSOR_02 | Pred: AERATION (1) | NA: 10.00% / A: 90.00%
[2026-03-20 17:14:58] | ID: SENSOR_03 | Pred: AERATION (1) | NA: 10.00% / A: 90.00%
[2026-03-20 17:15:00] | ID: SENSOR_01 | Pred: AERATION (1) | NA: 10.00% / A: 90.00%
[2026-03-20 17:15:00] | ID: SENSOR_02 | Pred: AER